# الفصل 12: عائلات المحوّلات: BERT وGPT وT5

> بعد أن بنينا معمارية Transformer الكاملة (Encoder-Decoder) في الفصل السابق، نكتشف في هذا الفصل حقيقة مثيرة: لم تَعُد كل النماذج الناجحة تستخدم المعمارية الكاملة! تفرّعت الأبحاث لاحقًا إلى ثلاثة مسارات رئيسية، كل...

- **الجزء:** 3 — المحوّلات الأساسية
- **الفصل:** 12
- **المستوى:** متقدم
- **زمن القراءة:** 25 دقيقة
- **المتطلبات:** الفصل 11: الترميز الموضعي والتطبيع وبنية المحوّل الكاملة
- **الوسوم:** BERT، GPT، T5، MLM

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch12-transformer-families.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-03-transformers%2Fch12-transformer-families.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-03-transformers/ch12-transformer-families.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch12-transformer-families.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch12-transformer-families.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch12-transformer-families.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch12-transformer-families.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-03-transformers%2Fch12-transformer-families.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-03-transformers/ch12-transformer-families.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch12-transformer-families.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch12-transformer-families.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-03-transformers/ch12-transformer-families.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> بعد أن بنينا معمارية Transformer الكاملة (Encoder-Decoder) في الفصل السابق، نكتشف في هذا الفصل حقيقة مثيرة: لم تَعُد كل النماذج الناجحة تستخدم المعمارية الكاملة! تفرّعت الأبحاث لاحقًا إلى ثلاثة مسارات رئيسية، كل...

> **بيانات الفصل**
>
> - **الجزء:** 3 — المحوّلات الأساسية
> - **المستوى:** متقدم
> - **زمن القراءة:** 25 دقيقة
> - **المتطلبات:** الفصل 11: الترميز الموضعي والتطبيع وبنية المحوّل الكاملة


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم GPT بالتفصيل
- فهم T5 بالتفصيل
- فهم BERT


## 1. لماذا يهمّك هذا الموضوع؟

بعد أن بنينا معمارية Transformer الكاملة (Encoder-Decoder) في الفصل السابق، نكتشف في هذا الفصل حقيقة مثيرة: **لم تَعُد كل النماذج الناجحة تستخدم المعمارية الكاملة!** تفرّعت الأبحاث لاحقًا إلى ثلاثة مسارات رئيسية، كل منها احتفظ بجزء من معمارية Transformer الأصلية وتخصص فيه:

- **BERT** (2018): احتفظ بـ **Encoder فقط**، وتخصص في الفهم (Understanding).
- **GPT** (2018 فصاعدًا): احتفظ بـ **Decoder فقط**، وتخصص في التوليد (Generation) — وهذا هو المسار الذي هيمن على عصر الـ LLMs الحديث بأكمله.
- **T5** (2019): احتفظ بـ **Encoder-Decoder الكامل**، ووحّد كل مهام NLP في صيغة "نص إلى نص" واحدة.

فهم هذه الثلاثة بعمق — لماذا اختار كل منها بنيته، وما أهداف التدريب المسبق (pretraining objectives) الخاصة به، وأين يتفوق كل منها — هو **حجر الأساس** لفهم كل نموذج لغوي حديث تقريبًا، بما فيها GPT-4 وLLaMA وClaude وGemini، التي جميعها (باستثناءات قليلة) سليلة مباشرة لعائلة Decoder-only.

---


## 2. الخريطة الذهنية


```text
Transformer الأصلي (Encoder + Decoder)
│
├── أخذ Encoder فقط ────────► عائلة BERT (Encoder-only)
│   │
│   ├── Bidirectional (كل كلمة ترى كل الكلمات، ماضيًا ومستقبلًا)
│   ├── هدف التدريب: Masked Language Modeling (MLM) + NSP
│   ├── لا يولّد نصًا تسلسليًا، بل يُنتج تمثيلات/تصنيفات
│   └── الاستخدام: تصنيف، استخراج معلومات، أسئلة-أجوبة استخراجية
│
├── أخذ Decoder فقط ────────► عائلة GPT (Decoder-only)
│   │
│   ├── Causal/Unidirectional (كل كلمة ترى فقط ما قبلها)
│   ├── هدف التدريب: Next Token Prediction (التنبؤ بالكلمة التالية)
│   ├── توليدي بطبيعته (autoregressive generation)
│   └── الاستخدام: توليد نص، محادثة، برمجة، كل تطبيقات LLM الحديثة
│
└── احتفظ بكليهما ──────────► عائلة T5 (Encoder-Decoder)
    │
    ├── Encoder يرى المدخل بالكامل (bidirectional) + Decoder يولّد (causal)
    ├── هدف التدريب: Span Corruption (إخفاء وإعادة توليد نطاقات نصية)
    ├── "Text-to-Text": كل مهمة NLP تُصاغ كتحويل نص إلى نص
    └── الاستخدام: ترجمة، تلخيص، أي مهمة تحويلية (input→output) واضحة
```


---


## 3. الشرح المتدرج

### 🟢 طبقة الحدس

تخيّل ثلاثة أنواع من القرّاء:

- **BERT** يشبه طالبًا يقرأ **فقرة كاملة أولًا** (رؤية كل الكلمات معًا، للأمام وللخلف)، ثم يُجيب عن أسئلة حول الفقرة أو يملأ فراغات داخلها بالاعتماد على السياق الكامل المحيط بالفراغ من الجهتين.

- **GPT** يشبه راويًا يكتب قصة **كلمة بكلمة من اليسار لليمين**، كل كلمة جديدة تعتمد فقط على ما كُتب قبلها، دون أي إمكانية "للغش" بالنظر لما سيُكتب لاحقًا (لأنه ببساطة لم يُكتب بعد).

- **T5** يشبه مترجمًا: يقرأ الجملة المصدر كاملة أولًا (كـ BERT)، ثم يُنتج الجملة الهدف تدريجيًا كلمة بكلمة (كـ GPT) مع النظر المستمر للجملة المصدر بالكامل أثناء الإنتاج.

### 🟡 طبقة الآلية: BERT بالتفصيل

**المعمارية**: Encoder فقط من Transformer الأصلي، مكدَّس 12 طبقة (BERT-base، 110M معامل) أو 24 طبقة (BERT-large، 340M معامل).

**هدف التدريب 1 — Masked Language Modeling (MLM)**:

نُخفي عشوائيًا 15% من الرموز في كل تسلسل مُدخَل، ونطلب من النموذج التنبؤ بها اعتمادًا على السياق الكامل (ثنائي الاتجاه):

$$\mathcal{L}_{MLM} = -\sum_{i \in \text{masked}} \log P(x_i | x_{\setminus \text{masked}})$$

تفصيل دقيق لاستراتيجية الإخفاء (لتقليل الفجوة بين التدريب المسبق والاستخدام الفعلي، حيث لا يوجد رمز `[MASK]` عند الاستدلال):
- 80% من الحالات: استبدال الرمز المُختار بـ `[MASK]`
- 10% من الحالات: استبداله برمز عشوائي آخر
- 10% من الحالات: تركه كما هو دون تغيير

**هدف التدريب 2 — Next Sentence Prediction (NSP)**:

يُعطى النموذج جملتين A وB، ويُطلب منه التنبؤ: هل B تتبع A فعليًا في النص الأصلي، أم أنها جملة عشوائية؟ (أُثبت لاحقًا في أوراق مثل RoBERTa أن NSP ذو فائدة محدودة، وتم التخلي عنه في نماذج لاحقة لصالح استراتيجيات أخرى).

**رمز `[CLS]`**: يُضاف في بداية كل تسلسل مُدخَل، وتمثيله النهائي (بعد كل طبقات Encoder) يُستخدم كـ"ملخص" لكامل التسلسل لمهام التصنيف.

### 🟡 GPT بالتفصيل

**المعمارية**: Decoder فقط من Transformer الأصلي — لكن بدون جزء Cross-Attention (لأنه لا يوجد Encoder لينتبه إليه)، فقط Masked Self-Attention + FeedForward مكدَّسة.

**هدف التدريب — Causal/Autoregressive Language Modeling**:

$$\mathcal{L}_{LM} = -\sum_{t=1}^{T} \log P(x_t | x_1, x_2, ..., x_{t-1})$$

ببساطة: تنبأ بالكلمة التالية اعتمادًا فقط على كل الكلمات **قبلها**. هذا الهدف البسيط جدًا — عندما يُدرَّب على كميات هائلة من النصوص (مئات المليارات إلى تريليونات الرموز) بمقاييس ضخمة — ينتج القدرات المذهلة التي نراها في GPT-3 وما بعده: التفكير المنطقي البسيط، الترجمة، التلخيص، البرمجة، كل ذلك **ناشئ (emergent)** من هذا الهدف الوحيد البسيط دون أي إشراف صريح لهذه المهارات تحديدًا.

**التوليد (Generation/Inference)**: يتم بشكل تسلسلي (autoregressive): نولّد رمزًا واحدًا، نُلحقه بالمدخل، ثم نولّد الرمز التالي بناءً على التسلسل الموسّع، وهكذا — هذا يفسّر لماذا توليد النص بطيء نسبيًا مقارنة بالفهم (كل رمز جديد يتطلب تمريرة أمامية كاملة عبر النموذج).

### 🟡 T5 بالتفصيل

**المعمارية**: Encoder-Decoder الكامل كما في الورقة الأصلية (مع تعديلات طفيفة مثل Relative Positional Bias بدلًا من PE المطلق).

**هدف التدريب — Span Corruption**: بدلًا من إخفاء رموز مفردة (كـ BERT)، يُخفي T5 **نطاقات متجاورة** من الرموز (spans) ويستبدلها برمز خاص (sentinel token) واحد، ثم يُدرَّب الـ Decoder على توليد النطاقات المفقودة بالترتيب:


```text
المدخل الأصلي:  "القطة الصغيرة تنام على السجادة الحمراء بهدوء"
بعد الإخفاء:    "القطة <X> تنام على <Y> بهدوء"
الهدف:          "<X> الصغيرة <Y> السجادة الحمراء <Z>"
```


**فلسفة "Text-to-Text" الموحَّدة**: الابتكار الأهم في T5 ليس المعمارية (مشابهة لـ Transformer الأصلي) بل **توحيد صياغة كل مهام NLP** كتحويل نص-إلى-نص، بإضافة بادئة (prefix) تصف المهمة:


```text
التصنيف:   "cola sentence: الجملة غير صحيحة نحويًا"  → "unacceptable"
الترجمة:    "translate English to German: Hello"       → "Hallo"
التلخيص:    "summarize: <فقرة طويلة>"                   → "<ملخص قصير>"
```


هذا التوحيد مهّد الطريق مفاهيميًا لفكرة الـ **Prompting** السائدة اليوم في كل LLM حديث.

### 🔴 طبقة التعمّق: الكود الكامل


In [ ]:
import torch
import torch.nn as nn
import math

# ============================================================
# 1. MiniBERT — نموذج Encoder-only مصغّر كامل
# ============================================================
class MiniBERT(nn.Module):
    def __init__(self, vocab_size, d_model=256, num_heads=8,
                 num_layers=6, d_ff=1024, max_len=512, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.segment_embedding = nn.Embedding(2, d_model)   # لجملة A أو B (NSP)
        self.position_embedding = nn.Embedding(max_len, d_model)  # Learned PE
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(d_model)

        self.encoder_layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        # رؤوس التدريب المسبق
        self.mlm_head = nn.Linear(d_model, vocab_size)
        self.nsp_head = nn.Linear(d_model, 2)

    def forward(self, input_ids, segment_ids, attention_mask):
        batch_size, seq_len = input_ids.shape
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)

        x = (self.token_embedding(input_ids)
             + self.segment_embedding(segment_ids)
             + self.position_embedding(positions))
        x = self.layer_norm(x)
        x = self.dropout(x)

        for layer in self.encoder_layers:
            x, _ = layer(x, attention_mask)

        # رمز [CLS] هو أول موضع، يُستخدم لـ NSP/التصنيف
        cls_representation = x[:, 0, :]

        mlm_logits = self.mlm_head(x)              # (batch, seq_len, vocab_size)
        nsp_logits = self.nsp_head(cls_representation)  # (batch, 2)

        return mlm_logits, nsp_logits

def mask_tokens_for_mlm(input_ids, vocab_size, mask_token_id,
                         special_token_ids, mlm_probability=0.15):
    """
    تطبيق استراتيجية إخفاء BERT: 80% [MASK], 10% عشوائي, 10% بلا تغيير.
    """
    labels = input_ids.clone()
    probability_matrix = torch.full(labels.shape, mlm_probability)

    # لا نُخفي الرموز الخاصة (CLS, SEP, PAD)
    special_mask = torch.isin(input_ids, torch.tensor(special_token_ids))
    probability_matrix.masked_fill_(special_mask, value=0.0)

    masked_indices = torch.bernoulli(probability_matrix).bool()
    labels[~masked_indices] = -100  # تجاهل الخسارة على الرموز غير المُخفاة

    # 80%: استبدال بـ [MASK]
    indices_replaced = torch.bernoulli(torch.full(labels.shape, 0.8)).bool() & masked_indices
    input_ids[indices_replaced] = mask_token_id

    # 10%: استبدال برمز عشوائي
    indices_random = (torch.bernoulli(torch.full(labels.shape, 0.5)).bool()
                       & masked_indices & ~indices_replaced)
    random_tokens = torch.randint(vocab_size, labels.shape, dtype=torch.long)
    input_ids[indices_random] = random_tokens[indices_random]

    # 10% المتبقية: تبقى كما هي (لا حاجة لفعل شيء)
    return input_ids, labels

# ============================================================
# 2. MiniGPT — نموذج Decoder-only مصغّر كامل (بالتوليد)
# ============================================================
class GPTBlock(nn.Module):
    """طبقة Decoder مبسّطة بدون Cross-Attention (لا يوجد Encoder)."""
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = PositionWiseFeedForward(d_model, d_ff, dropout)

    def forward(self, x, causal_mask):
        normed = self.ln1(x)
        attn_out, attn_weights = self.attn(normed, normed, normed, causal_mask)
        x = x + attn_out

        normed = self.ln2(x)
        ffn_out = self.ffn(normed)
        x = x + ffn_out

        return x, attn_weights

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=256, num_heads=8,
                 num_layers=6, d_ff=1024, max_len=512, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.max_len = max_len
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_len, d_model)
        self.dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            GPTBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        self.final_ln = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # ربط أوزان التضمين بطبقة الإخراج (weight tying) — تقنية قياسية لتقليل المعاملات
        self.lm_head.weight = self.token_embedding.weight

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)

        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        x = self.dropout(x)

        causal_mask = create_causal_mask(seq_len).to(input_ids.device)

        for block in self.blocks:
            x, _ = block(x, causal_mask)

        x = self.final_ln(x)
        logits = self.lm_head(x)   # (batch, seq_len, vocab_size)
        return logits

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=50, temperature=1.0, top_k=None):
        """
        توليد نص ذاتي الانحدار، رمزًا تلو الآخر.
        """
        self.eval()
        for _ in range(max_new_tokens):
            # اقتصاص السياق إذا تجاوز الطول الأقصى
            input_cropped = input_ids[:, -self.max_len:]

            logits = self.forward(input_cropped)
            # نأخذ فقط logits آخر موضع (التنبؤ بالكلمة التالية)
            next_token_logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(next_token_logits, top_k)
                next_token_logits[next_token_logits < v[:, [-1]]] = -float('inf')

            probs = torch.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            input_ids = torch.cat([input_ids, next_token], dim=1)

        return input_ids

# ============================================================
# 3. مفهوم T5 المبسّط — Span Corruption + Text-to-Text
# ============================================================
def create_span_corruption_example(tokens: list, noise_density: float = 0.15,
                                     mean_span_length: float = 3.0,
                                     sentinel_start_id: int = 32000):
    """
    تنفيذ مبسّط لاستراتيجية T5 Span Corruption:
    يختار نطاقات عشوائية متجاورة ويستبدلها برمز sentinel واحد.
    """
    num_tokens = len(tokens)
    num_to_mask = max(1, int(round(num_tokens * noise_density)))

    masked_input = []
    target = []
    sentinel_id = sentinel_start_id

    i = 0
    masked_count = 0
    while i < num_tokens:
        if masked_count < num_to_mask and np.random.random() < (noise_density / mean_span_length):
            span_length = min(
                np.random.poisson(mean_span_length) + 1,
                num_tokens - i
            )
            masked_input.append(sentinel_id)
            target.append(sentinel_id)
            target.extend(tokens[i:i + span_length])
            sentinel_id += 1
            masked_count += span_length
            i += span_length
        else:
            masked_input.append(tokens[i])
            i += 1

    target.append(sentinel_id)  # رمز إنهاء الهدف
    return masked_input, target

def format_as_text_to_text(task_name: str, input_text: str) -> str:
    """تنسيق المدخل بصيغة T5 الموحدة (بادئة المهمة + النص)."""
    task_prefixes = {
        "translation_ar_en": "translate Arabic to English: ",
        "summarization": "summarize: ",
        "sentiment": "sst2 sentence: ",
        "cola": "cola sentence: ",
    }
    prefix = task_prefixes.get(task_name, "")
    return prefix + input_text


---


## 4. مثال محلول خطوة بخطوة

لنُطبّق MLM على جملة عربية بسيطة: **"الذكاء الاصطناعي يغيّر العالم"**، مُرمَّزة إلى الرموز الفرعية: `["ال", "ذكاء", "الاصطناعي", "يغير", "العالم"]`.

**خطوة الإخفاء** (بافتراض اختيار كلمة "يغير" للإخفاء بنسبة 15%):

$$\text{المدخل الأصلي}: [\text{"ال", "ذكاء", "الاصطناعي", "يغير", "العالم"}]$$
$$\text{المدخل بعد MLM (80% حالة)}: [\text{"ال", "ذكاء", "الاصطناعي", [\text{MASK}], "العالم"}]$$

النموذج يستقبل هذا التسلسل بالكامل عبر Encoder ثنائي الاتجاه، وحالة `[MASK]` في طبقة الإخراج تحاول التنبؤ بالاحتمالات على كامل المفردات. بما أن النموذج **يرى السياق على الجهتين** ("الاصطناعي" قبلها و"العالم" بعدها)، فمن المرجح جدًا أن يُعطي احتمالًا عاليًا لكلمات مثل "يغير" أو "يُحدث" أو "يُعيد تشكيل" — جميعها متوافقة دلاليًا مع السياق الكامل.

**مقارنة مع GPT في نفس الموضع**: لو كان هذا نموذج GPT (Causal)، فعند التنبؤ بالكلمة في نفس الموضع، لن يرى **إلا** `["ال", "ذكاء", "الاصطناعي"]` (بدون "العالم" التي تأتي بعدها) — هذا يجعل المهمة أصعب وأكثر غموضًا من منظور GPT، لأن معلومة "العالم" (المفعول به) كانت ستُضيّق الاحتمالات بشكل كبير. هذا الفرق بالضبط هو جوهر الاختلاف بين **الفهم ثنائي الاتجاه** (BERT) و**التوليد أحادي الاتجاه** (GPT).

---


## 5. نصائح وحيل

> 💡 **اختر العائلة حسب طبيعة المهمة وليس حسب الشهرة**: إذا كانت مهمتك تصنيفًا أو استخراج معلومات من نص ثابت معروف بالكامل (ليس توليديًا)، فإن نموذجًا من عائلة BERT (أو مشتقاتها الحديثة مثل RoBERTa/DeBERTa) غالبًا أكفأ حسابيًا وأدق من استخدام LLM توليدي ضخم لنفس المهمة.

> 💡 **Weight Tying تقنية بسيطة وفعّالة**: ربط أوزان طبقة تضمين المدخل بطبقة الإسقاط النهائية (`lm_head.weight = token_embedding.weight`) يُقلّل عدد المعاملات بشكل ملحوظ (قد يصل الفرق لملايين المعاملات في مفردات كبيرة) دون الإضرار بالأداء، وهي شائعة في كل من GPT وT5.

> 💡 **درجة الحرارة (temperature) وTop-k/Top-p أدوات تحكم أساسية بالتوليد**: `temperature` منخفضة (مثل 0.3) تُنتج نصًا أكثر "حتمية" وتماسكًا لكن أقل تنوعًا؛ `temperature` عالية (مثل 1.2) تزيد التنوع لكن تزيد مخاطر عدم التماسك — Top-k وTop-p (nucleus sampling) يُقيّدان مساحة الاختيار لتفادي الرموز شديدة الندرة وغير المنطقية.

> 💡 **T5 مفيد جدًا عندما تكون العلاقة input→output واضحة ومحددة البنية**: مثل الترجمة أو التلخيص المُقيَّد، لأن بنية Encoder-Decoder الصريحة (مع Cross-Attention) تمنح فصلًا واضحًا بين "فهم المصدر" و"توليد الهدف" — غالبًا أكفأ من استخدام Decoder-only ضخم لنفس هذه المهام المحددة جيدًا.

---


## 6. أخطاء شائعة ومصائد

> ⚠️ **استخدام BERT للتوليد النصي**: BERT ليس نموذجًا توليديًا بطبيعته (لا آلية ذاتية الانحدار فيه، ولا حتى ترتيب توليد محدد) — محاولة "توليد" نص منه (مثلًا بالتنبؤ المتكرر بكلمات مخفية) تُعطي نتائج ضعيفة جدًا مقارنة باستخدام نموذج GPT أو T5 المصمَّم خصيصًا لذلك.

> ⚠️ **نسيان Causal Mask في تدريب GPT**: إذا نسيت تطبيق القناع السببي، فإن النموذج "يغش" برؤية الكلمات المستقبلية أثناء التدريب (data leakage صريح)، مما يُنتج خسارة تدريب منخفضة بشكل مزيف لكن أداء كارثي عند التوليد الفعلي حيث لا توجد كلمات مستقبلية لرؤيتها.

> ⚠️ **الخلط بين NSP وMLM من حيث الأهمية**: أظهرت أبحاث لاحقة (RoBERTa 2019) أن حذف NSP بالكامل والتركيز فقط على MLM مع تدريب أطول وبيانات أكثر يُحسّن الأداء — قد يُفاجَأ من يفترض أن كل مكونات الورقة الأصلية ضرورية بالتساوي.

> ⚠️ **تجاهل بادئات المهام (task prefixes) عند استخدام T5**: نسيان إضافة البادئة الصحيحة (مثل `"translate English to German: "`) يُربك النموذج تمامًا حول المهمة المطلوبة، لأن T5 دُرِّب على التمييز بين المهام **فقط** عبر هذه البادئات النصية.

---


## 7. جدول مقارنة

| الخاصية | BERT | GPT | T5 |
|---|---|---|---|
| **الجزء المُستخدَم من Transformer** | Encoder فقط | Decoder فقط | Encoder + Decoder كاملان |
| **اتجاه الانتباه** | ثنائي الاتجاه (Bidirectional) | أحادي الاتجاه (Causal) | Encoder ثنائي + Decoder أحادي |
| **هدف التدريب المسبق** | MLM + NSP | Next Token Prediction | Span Corruption |
| **طبيعة المهمة الأساسية** | فهم (Understanding) | توليد (Generation) | تحويل نص-إلى-نص (شامل) |
| **التوليد التسلسلي** | غير مناسب بطبيعته | طبيعي وأساسي | طبيعي عبر الـ Decoder |
| **أمثلة الاستخدام المثالية** | تصنيف، NER، QA استخراجي | محادثة، كتابة إبداعية، برمجة | ترجمة، تلخيص، تحويلات محددة |
| **أمثلة نماذج** | BERT, RoBERTa, DeBERTa, ELECTRA | GPT-2/3/4, LLaMA, Claude, Gemini | T5, mT5, Flan-T5, BART (مشابه) |
| **الهيمنة في عصر LLM الحالي** | محدودة (مهام متخصصة) | شبه كاملة | متخصصة (ترجمة/تلخيص تحديدًا) |
| **كفاءة الاستدلال لمهمة تصنيف واحدة** | عالية جدًا (تمريرة واحدة) | أقل كفاءة (قد يحتاج توليدًا) | متوسطة (يحتاج Encoder+Decoder) |

---


## 8. ملخص أفضل الممارسات

- ✅ اختر عائلة النموذج بناءً على طبيعة المهمة: فهم بحت → BERT-like؛ توليد حر → GPT-like؛ تحويل مُقيَّد input→output → T5-like.
- ✅ استخدم Weight Tying بين طبقة التضمين وطبقة الإخراج في النماذج التوليدية لتقليل المعاملات دون خسارة أداء.
- ✅ تذكّر دائمًا أن القناع السببي إلزامي في تدريب أي نموذج من عائلة GPT.
- ✅ عند استخدام T5 أو مشتقاته، التزم بصيغة بادئات المهام التي دُرِّب عليها النموذج بالضبط.
- ✅ لا تفترض أن النموذج الأكبر أو الأحدث هو الأنسب دائمًا — نموذج BERT صغير مُدرَّب جيدًا قد يتفوق على LLM ضخم في مهمة تصنيف محددة من حيث الدقة والكفاءة الحسابية معًا.

---


## 9. أسئلة مقابلات

**س1: لماذا هيمنت معمارية Decoder-only (عائلة GPT) على عصر النماذج اللغوية الكبيرة الحديثة، رغم أن BERT وT5 ظهرا بنفس الفترة تقريبًا وأظهرا نتائج قوية جدًا في مهامهما؟**

**الإجابة النموذجية**: هناك عدة أسباب متضافرة: (1) **البساطة والتوحيد**: هدف "التنبؤ بالكلمة التالية" بسيط جدًا للتوسع (scale) على بيانات نصية خام ضخمة جدًا دون الحاجة لأي هندسة خاصة بالمهمة (كما تتطلب MLM استراتيجيات إخفاء معقدة، وSpan Corruption احتياجات تصميم إضافية)؛ (2) **المرونة الشاملة**: نموذج توليدي واحد مُدرَّب جيدًا يمكنه، عبر الـ Prompting فقط دون أي إعادة تدريب، أداء أي مهمة تقريبًا (تصنيف، تلخيص، ترجمة، إجابة أسئلة، برمجة) بصياغتها كمهمة توليد نص — وهذا أثبت قابلية توسع هائلة (الظواهر الناشئة/emergent abilities) مع زيادة الحجم؛ (3) **الخصائص قليلة اللقطات (Few-shot)**: اكتشفت ورقة GPT-3 أن النماذج التوليدية الكبيرة جدًا يمكنها التعلم من أمثلة قليلة ضمن الـ Prompt فقط (in-context learning) دون أي تحديث لأوزانها، وهي قدرة لم تظهر بنفس القوة في BERT أو T5. هذه العوامل مجتمعة جعلت الاستثمار الهائل في تكبير نماذج Decoder-only (من GPT-2 إلى GPT-4 وما بعده) الاستراتيجية الأكثر عائدًا على الاستثمار البحثي والتجاري.

**س2: ما الفرق الجوهري بين هدف التدريب MLM في BERT وSpan Corruption في T5؟**

**الإجابة النموذجية**: كلاهما يعتمدان على فكرة "إخفاء ثم إعادة بناء"، لكن MLM يُخفي **رموزًا مفردة متفرقة** (قد تكون غير متجاورة إطلاقًا) ويتنبأ بها **في نفس المواضع** عبر رأس تصنيف مُطبَّق على كامل التسلسل دفعة واحدة (غير تسلسلي/non-autoregressive) — وهذا مناسب لـ Encoder فقط حيث لا يوجد توليد تسلسلي. أما Span Corruption في T5، فيُخفي **نطاقات متجاورة كاملة** (قد تمتد لعدة كلمات متتالية) ويستبدل كل نطاق برمز sentinel واحد فقط، ثم يُطلب من الـ Decoder توليد **تسلسل الأهداف كاملًا بشكل ذاتي الانحدار** (رمز تلو الآخر) بدءًا من الـ sentinel الأول — وهذا يستفيد من وجود Decoder توليدي فعلي في المعمارية، ويجعل المهمة أقرب لمهام الترجمة/التلخيص الفعلية التي سيُستخدَم فيها النموذج لاحقًا، وأكفأ حسابيًا أيضًا من MLM لأن الهدف (target sequence) أقصر بكثير من التسلسل الأصلي الكامل.

**س3: كيف يُمكن لنموذج من عائلة BERT أن يُستخدَم في الإجابة عن الأسئلة (QA) رغم أنه ليس نموذجًا توليديًا؟**

**الإجابة النموذجية**: تُصاغ مهمة الإجابة عن الأسئلة مع BERT كمهمة **استخراجية (extractive)** وليست توليدية: يُعطى النموذج الفقرة (context) والسؤال معًا كتسلسل واحد مُدخَل (مفصولَين برمز `[SEP]`)، ويُضاف رأسان خطيان صغيران فوق تمثيلات Encoder النهائية لكل موضع في الفقرة — أحدهما يتنبأ باحتمال أن هذا الموضع هو **بداية** الإجابة، والآخر يتنبأ باحتمال أنه **نهاية** الإجابة. أثناء الاستدلال، نختار الموضعين صاحبي أعلى احتمال بداية/نهاية (مع قيد أن النهاية بعد البداية)، ونستخرج النص الفعلي بينهما من الفقرة الأصلية كإجابة. هذا يختلف جوهريًا عن QA التوليدي (كما في GPT حيث يُولَّد نص الإجابة حرفيًا رمزًا تلو الآخر، وقد لا يكون موجودًا حرفيًا في النص المصدر).

**س4: ماذا يعني "Weight Tying" ولماذا يُستخدَم في النماذج التوليدية؟**

**الإجابة النموذجية**: Weight Tying هو ربط مصفوفة أوزان طبقة تضمين المُدخَل (token embedding، بحجم `vocab_size × d_model`) بمصفوفة أوزان طبقة الإسقاط النهائية التي تُحوّل تمثيل الحالة المخفية الأخيرة إلى احتمالات على المفردات (`lm_head`، بحجم `d_model × vocab_size` أي المصفوفة المنقولة لنفس الحجم). الحدس وراء ذلك: كلا الطبقتين تُجريان نوعًا من "الترجمة" بين فضاء الرموز المتقطع (الكلمات) وفضاء التمثيل المستمر (المتجهات)، ومن المنطقي أن تتشارك نفس الهندسة الدلالية (كلمة ذات تضمين قريب من كلمة أخرى يجب أن يكون احتمال التنبؤ بها أيضًا متشابهًا في سياقات مشابهة). عمليًا، يُقلّل هذا الربط عدد المعاملات الكلي بشكل كبير جدًا (قد يُشكّل ملايين أو حتى عشرات الملايين من المعاملات في مفردات كبيرة الحجم مع d_model كبير)، وأظهرت التجارب أنه لا يُضعف الأداء بل قد يُحسّنه أحيانًا بفضل تقليل خطر الحفظ الزائد (overfitting).

---


## 10. تمرين عملي

1. درّب `MiniBERT` مصغّرًا جدًا (d_model=64، طبقتان فقط) على مجموعة جمل عربية قصيرة بهدف MLM فقط (بدون NSP)، وتحقق من أن النموذج يتعلم التنبؤ بكلمات محذوفة منطقية في جمل بسيطة.
2. درّب `MiniGPT` بنفس الحجم التقريبي على نفس النصوص بهدف Next Token Prediction، ثم استخدم دالة `generate` لتوليد استكمال لجملة بداية قصيرة — قارن جودة التوليد بدرجات حرارة مختلفة (0.3، 0.7، 1.2).
3. نفّذ `create_span_corruption_example` على فقرة عربية من 20 كلمة تقريبًا، واطبع المدخل المُخفى والهدف الناتج للتحقق من صحة التطبيق.
4. صِغ 3 مهام NLP مختلفة (تصنيف، ترجمة مبسّطة، تلخيص) بصيغة Text-to-Text مع بادئات مناسبة، كما يفعل T5.

---


## 11. مراجع للتعمّق

- Devlin, J., et al. (2018). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. NAACL.
- Radford, A., et al. (2018/2019). *Improving Language Understanding by Generative Pre-Training* (GPT) و *Language Models are Unsupervised Multitask Learners* (GPT-2). OpenAI.
- Brown, T., et al. (2020). *Language Models are Few-Shot Learners* (GPT-3). NeurIPS.
- Raffel, C., et al. (2019). *Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer* (T5). JMLR.
- Liu, Y., et al. (2019). *RoBERTa: A Robustly Optimized BERT Pretraining Approach*.


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — المحوّلات الأساسية / الفصل 12: عائلات المحوّلات: BERT وGPT وT5.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
